In [ ]:
import torch
from PIL import Image
import os
import numpy as np
from transformers import CLIPProcessor, CLIPModel
import matplotlib.pyplot as plt
import yaml
from cleanfid import fid
import lpips
import torchvision.transforms as T
import itertools

with open(os.path.expanduser("/work/cvcs2026/stochastic_parrots/config.yaml"), "r") as f:
    config = yaml.safe_load(f)

# Definition Output Directories

In [ ]:
MODEL_PATH = config["paths"]["base_model_dir"]
LORA_PATH = config["paths"]["lora_model_dir"]
LORA_WEIGHTS_FILE = config["weights_file"]["lora_weights"]

REF_IMAGES_DIR = config["paths"]["instance_images_dir"]

OUTPUT_CLIPI_SDXL = config["paths"]["evaluation_dir"] + "/metrics/fidelity/images_sdxl"
OUTPUT_CLIPI_LORA = config["paths"]["evaluation_dir"] + "/metrics/fidelity/images_lora"

OUTPUT_CLIPT_SDXL = config["paths"]["evaluation_dir"] + "/metrics/preservability/clipt/images_sdxl"
OUTPUT_CLIPT_LORA = config["paths"]["evaluation_dir"] + "/metrics/preservability/clipt/images_lora"

OUTPUT_FID_SDXL = config["paths"]["evaluation_dir"] + "/metrics/preservability/fid/images_sdxl"
OUTPUT_FID_LORA = config["paths"]["evaluation_dir"] + "/metrics/preservability/fid/images_lora"

OUTPUT_LPIPS_SDXL = config["paths"]["evaluation_dir"] + "/metrics/preservability/lpips/images_sdxl"
OUTPUT_LPIPS_LORA = config["paths"]["evaluation_dir"] + "/metrics/preservability/lpips/images_lora"


model_id = "openai/clip-vit-large-patch14"  # Model version: ViT Large (24 layers) with patch14 (each patch is 14x14 pixels)
model = CLIPModel.from_pretrained(model_id).to("cuda")  # Load the CLIP model (pretrained) and move it to the specified device (GPU or CPU)
processor = CLIPProcessor.from_pretrained(model_id)  # Load the CLIP processor to automatically preprocess images and text for the model

# CLIP-I Metric
Test for "Personalization Quality". How well the model captures the target concept? 

We use this metric to measure the distance between, the generated images and the reference images.

## Helper Functions

In [ ]:
def get_features(image_paths):
    images = [Image.open(p).convert("RGB") for p in image_paths]  # Load and convert images to RGB (to be sure)
    inputs = processor(images=images, return_tensors="pt").to("cuda") # Transform images into tensors (pytorch) and move to DEVICE
    
    with torch.no_grad():
        outputs = model.get_image_features(**inputs) # Get the image features from the CLIP model (vector of 1024 numbers)
        
        # 1. Extract the tensor from the model output (it can be a tensor diretly or a BaseModelOutputWithPooling, depends on the version of model)
        if torch.is_tensor(outputs):
            features = outputs
        else:
            # If it's a BaseModelOutputWithPooling, we look for the right field
            # In CLIP it's usually called 'image_embeds' or it's the first element
            features = getattr(outputs, "image_embeds", outputs[0])

        # 2. If the model output is 3D (Batch, Patches, Features), it outputs the features of all the patches. We need only "CLS Token" (the first patch)
        if len(features.shape) == 3:
            features = features[:, 0, :]

        # 3. Normalize the features to have unit norm (this is important for cosine similarity)
        features = features / features.norm(p=2, dim=-1, keepdim=True)
        
    return features


# Create a "footprint" of our reference images by averaging their features. This will be our "reference feature" to compare against generated images.

ref_paths = [os.path.join(REF_IMAGES_DIR, f) for f in os.listdir(REF_IMAGES_DIR) if f.endswith(('.png', '.jpg'))] # Get all image paths in the reference directory
ref_features = get_features(ref_paths)  # Get the features of all reference images (matrix of size [num_ref_images, feature_dim])
avg_ref_feature = ref_features.mean(dim=0, keepdim=True)  # Average the features to get a single "reference feature" (vector of size [1, feature_dim])
avg_ref_feature /= avg_ref_feature.norm(p=2, dim=-1, keepdim=True) # Normalize the average reference feature to have unit norm (important for cosine similarity)


# This function evaluates the fidelity of generated images by computing the cosine similarity. It returns the mean and std of the similarities for all generated images
def evaluate_folder(folder_path):
    gen_paths = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.png')]  # Get all generated image paths in the specified folder
    if not gen_paths: return 0
    
    all_similarities = []
    for path in gen_paths:
        gen_feat = get_features([path])  # Get the feature of the generated image (vector of size [1, feature_dim])
        sim = (gen_feat @ avg_ref_feature.T).item()  # Compute the cosine similarity between the generated image feature and the average reference feature (scalar)
        all_similarities.append(sim) # Append the similarity to the list of all similarities
    
    return np.mean(all_similarities), np.std(all_similarities)  # Return mean and std of the similarities for all generated images in the folder

## Execution

In [ ]:
#print("Compute similarity for Base folder...")
mean_lora, std_lora = evaluate_folder(OUTPUT_CLIPI_SDXL)

#print("Compute similarity for LoRA folder...")
mean_base, std_base = evaluate_folder(OUTPUT_CLIPI_LORA)

print("\n--- RESULTS SUBJECT FIDELITY ---")
print(f"{'Model':<25} | {'Mean':<15} | {'Std':<15}")

print(f"{'Base':<25} | {mean_base:<15.4f} | {std_base:<15.4f}")
print(f"{'LoRA':<25} | {mean_lora:<15.4f} | {std_lora:<15.4f}")

print(f"Fidelity increment: {((mean_lora - mean_base) / mean_base) * 100:.2f}%")

# CLIP-T Metric
Test for "Language Drift". How well the model understands the "non-cat" prompts.

We use this metric to measure the distance between the generated images and their prompt

## Helper Functions

In [ ]:
prompts = [
    "A photo of a black cat",
    "A photo of a dog",
    "A photo of a rabbit",
    "A photo of a fox",
    "A photo of a lion",
    "A photo of a tiger",
    "A mountain landscape",
    "A portrait of a person",
    "A city skyline",
    "A bowl of fruit",
    "A car on a road",
    "A cake on a table",
    "A beach at sunset",
    "A forest in autumn",
    "A space scene with planets",
    "A close-up of a flower",
    "A group of people at a party",
    "A bird flying in the sky",
    "A boat on a lake",
    "A plant in a pot",
    "A group of airballoons in the sky",
    "A robot in a futuristic city",
    "A cute 3D animated character in a fantasy setting",
    "A vintage car parked on a street in an old European city",
    "A living room interior"
]

In [ ]:
def compute_clip_t(image_path, text_prompt):
    img = Image.open(image_path).convert("RGB")
    
    # Processor accepts both text and images, and returns tensors ready for the model
    inputs = processor(
        text=[text_prompt], 
        images=[img], 
        return_tensors="pt", 
        padding=True
    ).to("cuda")
    
    with torch.no_grad():
        outputs = model(**inputs)
        
        # We extract the image and text embeddings from the model's output
        image_embeds = outputs.image_embeds
        text_embeds = outputs.text_embeds
        
        image_embeds /= image_embeds.norm(p=2, dim=-1, keepdim=True)
        text_embeds /= text_embeds.norm(p=2, dim=-1, keepdim=True)
        
        # Cosine similarity is computed as the dot product of the normalized embeddings
        similarity = (text_embeds @ image_embeds.T).item()
        
    return similarity


def evaluate_clipt(folder_path, nprompt):
    files = []
    p = prompts[nprompt]
    n = nprompt+1
    if n < 10: n = f"0{n}"
    for f in os.listdir(folder_path):
        if f.startswith(f"image_{n}"):
            files.append(os.path.join(folder_path, f))

    scores = []
    for file in files:
        score = compute_clip_t(file, p)
        scores.append(score)

    return np.mean(scores), np.std(scores)

## Execution

In [ ]:
results = []
print(f"{'Prompt':<25} | {'Base':<15} | {'LoRA':<15} | {'Drift (%)':<10}")
print("-" * 75)
for i, prompt in enumerate(prompts):
    mean_base, std_base = evaluate_clipt(OUTPUT_CLIPT_SDXL, i)
    mean_lora, std_lora = evaluate_clipt(OUTPUT_CLIPT_LORA, i)

    current_drift = (1 - (mean_lora / mean_base)) * 100 if mean_base != 0 else 0

    results.append({
        "prompt": prompt,
        "base": mean_base,
        "lora": mean_lora,
        "drift": current_drift
    })

    print(f"{prompt[:25]:<25} | {mean_base:.4f} ± {std_base:.4f} | {mean_lora:.4f} ± {std_lora:.4f} | {current_drift:.2f}%")

avg_base = np.mean([r["base"] for r in results])
avg_lora = np.mean([r["lora"] for r in results])
avg_drift = (1 - (avg_lora / avg_base)) * 100

print("-"*75)
print(f"Average CLIP-T Base: {avg_base:.4f}, Average CLIP-T LoRA: {avg_lora:.4f}, Average Drift: {avg_drift:.2f}%")

# FID Metric
Test for "Class Drift" and "Diversity". How well the specialized model can generate other images ("non TOK cat"). 

We use this metric to measure how different is the distribution of specialized model respect to the distribution of base model

NOTE: The more value is close to 0, the more similar the two sets of images are

## Helper Functions

In [ ]:
outputDirSdxl_cat = OUTPUT_FID_SDXL + "/cat"
outputDirLora_cat = OUTPUT_FID_LORA + "/cat"

outputDirSdxl_noncat = OUTPUT_FID_SDXL + "/noncat"
outputDirLora_noncat = OUTPUT_FID_LORA + "/noncat"

## Execution

In [ ]:
# Compute FID
score_fid_cat = fid.compute_fid(outputDirSdxl_cat, outputDirLora_cat, device=torch.device("cuda"))
score_fid_noncat = fid.compute_fid(outputDirSdxl_noncat, outputDirLora_noncat, device=torch.device("cuda"))

print(f"FID Score (Cat): {score_fid_cat:.4f}")
print(f"FID Score (Non-Cat): {score_fid_noncat:.4f}")

# LPIPS Metric
Test for "Mode Collapse". How different are the images generated by a model?

We use this metric to compare each image with the others, and compute how different they are

## Helper functions

In [ ]:
loss_fn_alex = lpips.LPIPS(net='alex').to("cuda")

def load_and_preprocess(path):
    img = Image.open(path).convert("RGB")
    transform = T.Compose([
        T.Resize((256, 256)),
        T.ToTensor(),
        T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) 
    ])
    return transform(img).unsqueeze(0).to("cuda")

def compute_internal_diversity(folder_path):
    files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.png')]
    if len(files) < 2: return 0
    
    distances = []
    
    for path_a, path_b in itertools.combinations(files, 2):
        img_a = load_and_preprocess(path_a)
        img_b = load_and_preprocess(path_b)
        
        with torch.no_grad():
            dist = loss_fn_alex(img_a, img_b)
            distances.append(dist.item())
            
    return sum(distances) / len(distances)

## Execution

In [ ]:
div_base = compute_internal_diversity(OUTPUT_LPIPS_SDXL)
div_lora = compute_internal_diversity(OUTPUT_LPIPS_LORA)

print(f"Internal Diversity (LPIPS) Base: {div_base:.4f}")
print(f"Internal Diversity (LPIPS) LoRA: {div_lora:.4f}")